# Model 07: genealogical ancestry versus genetic ancestry

A genealogical ancestor is connected to a descendant by parent-child relationships. A **genetic ancestor** is a genealogical ancestor who also contributed DNA that survives in the descendant.

This notebook separates three quantities: expected DNA fraction, probability of any autosomal DNA, and probability of a segment above a chosen cM threshold.

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = None
if IN_COLAB:
    REPO_DIR = Path('/content/Evolution-Creation')
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git','clone','-q','https://github.com/vafaei-ar/Evolution-Creation.git',str(REPO_DIR)],check=True)
    else:
        subprocess.run(['git','-C',str(REPO_DIR),'fetch','-q','origin','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'checkout','-q','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[dev]'],check=True)
else:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / 'src' / 'evolution_creation').exists():
            REPO_DIR = candidate
            break

if REPO_DIR is not None:
    src_path = str(REPO_DIR / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

print('Environment ready:', REPO_DIR if REPO_DIR is not None else 'using installed Python environment')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from evolution_creation.genetic_ancestry import (
    balanced_autosome_map,
    coop_genetic_ancestor_probability,
    expected_genetic_ancestor_count,
    simulate_path_replicates,
    simulate_segment_history,
)


## Published analytic benchmark

For the human approximation used by Agranat-Tamir, Mooney, and Rosenberg (2024), the mean number of fragments from a specific ancestor k generations back is

$$\lambda_k=\frac{22+33(k-1)}{2^{k-1}},$$

and the probability of at least one inherited autosomal fragment is $p_k=1-e^{-\lambda_k}$ for k >= 2, with $p_1=1$.

In [ ]:
max_generations=widgets.IntSlider(value=18,min=4,max=25,step=1,description='Generations')
replicates=widgets.IntSlider(value=1500,min=100,max=5000,step=100,description='Replicates')
threshold_cm=widgets.FloatSlider(value=6.0,min=0.0,max=20.0,step=0.5,description='Threshold cM')
crossovers=widgets.FloatSlider(value=33.0,min=20.0,max=50.0,step=1.0,description='Map Morgans')
seed=widgets.IntText(value=20260920,description='Seed')
display(max_generations,replicates,threshold_cm,crossovers,seed)

In [ ]:
def run_model(_=None):
    g=max_generations.value
    chromosome_map=balanced_autosome_map(22,crossovers.value)
    summary=simulate_path_replicates(
        max_generations=g,
        chromosome_lengths_morgans=chromosome_map,
        detectable_threshold_cm=threshold_cm.value,
        replicates=replicates.value,
        seed=seed.value,
    )
    x=summary.generations
    analytic=np.array([coop_genetic_ancestor_probability(int(k),22,crossovers.value) for k in x])
    fig,ax=plt.subplots(figsize=(10,5))
    ax.plot(x,analytic,linestyle='--',label='Poisson-fragment approximation')
    ax.plot(x,summary.any_dna_probability,label='simulation: any founder DNA')
    ax.plot(x,summary.detectable_probability,label=f'simulation: >= {threshold_cm.value:g} cM')
    ax.set(xlabel='Generations from ancestor to descendant',ylabel='Probability',ylim=(0,1.02))
    ax.legend(); plt.show()

    expected_fraction=0.5**x
    fig,ax=plt.subplots(figsize=(10,5))
    ax.semilogy(x,expected_fraction,linestyle='--',label='2^-k expectation')
    ax.semilogy(x,summary.mean_founder_fraction,label='simulation mean')
    ax.set(xlabel='Generations',ylabel='Mean fraction of diploid autosomal genome')
    ax.legend(); plt.show()

    genealogical=2.0**x
    genetic=np.array([expected_genetic_ancestor_count(int(k),22,crossovers.value) for k in x])
    fig,ax=plt.subplots(figsize=(10,5))
    ax.semilogy(x,genealogical,label='genealogical pedigree slots')
    ax.semilogy(x,genetic,label='expected genetic ancestors')
    ax.set(xlabel='Generations back',ylabel='Count')
    ax.legend(); plt.show()

    for k in [8,10,12,16]:
        if k<=g:
            i=k-1
            print(f'k={k:2d}: analytic any-DNA={analytic[i]:.4f}, simulated any-DNA={summary.any_dna_probability[i]:.4f}, detectable={summary.detectable_probability[i]:.4f}')

button=widgets.Button(description='Run simulation',button_style='primary')
button.on_click(run_model)
display(button)
run_model()

## Watch one chromosome history

The viewer below follows one specified genealogical path. Orange intervals are autosomal segments inherited from the focal ancestor. The partner in every later generation is assumed to carry no DNA from that focal ancestor.

In [ ]:
history_seed=widgets.IntText(value=20260922,description='History seed')
history_generation=widgets.IntSlider(value=1,min=1,max=18,description='Generation')
display(history_seed,history_generation)

def show_history(generation=1, history_seed=20260922):
    chromosome_map=balanced_autosome_map(22,crossovers.value)
    history=simulate_segment_history(max_generations=max_generations.value,chromosome_lengths_morgans=chromosome_map,seed=history_seed)
    generation=min(generation,len(history.segments_by_generation))
    segments=history.segments_by_generation[generation-1]
    fig,ax=plt.subplots(figsize=(11,8))
    for idx,(chromosome,length) in enumerate(zip(segments,history.chromosome_lengths_morgans),start=1):
        ax.plot([0,length],[idx,idx],linewidth=7,alpha=.18)
        for start,end in chromosome:
            ax.plot([start,end],[idx,idx],linewidth=7)
    ax.set(xlabel='Genetic-map position (Morgans)',ylabel='Autosome',title=f'Founder-derived segments at generation {generation}')
    ax.set_yticks(range(1,23)); ax.invert_yaxis(); plt.show()

widgets.interactive(show_history,generation=history_generation,history_seed=history_seed)

## Interpretation limits

This segment simulation follows **one** genealogical path. A distant ancestor can occur through multiple paths because of pedigree collapse. Multiple paths can increase the chance that some DNA survives, and those paths are not independent. Therefore the single-path curve should not be used by itself to infer the DNA contribution of a proposed universal historical ancestor.